# CWAB - Compressed Window Attention Broadcast
**Honest Benchmark. No cherry-picking.**

Author: Vladimir0-1 | License: MIT

In [ ]:
# Setup
!pip install torch transformers numpy matplotlib tqdm datasets -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
import gc
import warnings
import logging
import os

warnings.filterwarnings("ignore")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# Create CWAB class directly
class CWAB(nn.Module):
    def __init__(self, hidden_size, num_heads=8, window_size=512, num_global_tokens=64, dropout=0.1, short_seq_threshold=1024):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.window_size = min(window_size, hidden_size)
        self.num_global_tokens = num_global_tokens
        self.short_seq_threshold = short_seq_threshold
        self.global_memory = nn.Parameter(torch.randn(1, num_global_tokens, hidden_size))
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.compressor = nn.Conv1d(hidden_size, hidden_size, kernel_size=4, stride=4)
        self.mix_gate = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.Sigmoid())
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        batch, seq, dim = x.shape
        if seq <= self.short_seq_threshold:
            return self._full_attention(x)
        local_out = self._sliding_window(x)
        global_out = self._global_context(x)
        mix = self.mix_gate(torch.cat([local_out, global_out], dim=-1))
        out = mix * local_out + (1 - mix) * global_out
        return self.out_proj(self.dropout(out))

    def _sliding_window(self, x):
        batch, seq, dim = x.shape
        window = min(self.window_size, seq)
        if seq <= window:
            return self._full_attention(x)
        pad = (window - seq % window) % window
        x_pad = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
        padded_seq = x_pad.shape[1]
        n_windows = padded_seq // window
        windows = x_pad.reshape(batch, n_windows, window, dim)
        windows = windows.reshape(batch * n_windows, window, self.num_heads, self.head_dim)
        windows = windows.transpose(1, 2)
        attn = torch.matmul(windows, windows.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, windows)
        out = out.transpose(1, 2).reshape(batch * n_windows, window, dim)
        out = out.reshape(batch, n_windows, window, dim)
        out = out.reshape(batch, padded_seq, dim)
        return out[:, :seq, :] if pad > 0 else out

    def _full_attention(self, x):
        batch, seq, dim = x.shape
        q = self.q_proj(x).reshape(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        return out.transpose(1, 2).reshape(batch, seq, dim)

    def _global_context(self, x):
        batch, seq, dim = x.shape
        if seq >= 4 and self.num_global_tokens > 0:
            compressed = self.compressor(x.transpose(1, 2)).transpose(1, 2)
            compressed = compressed[:, :self.num_global_tokens, :]
        else:
            compressed = x
        memory = self.global_memory.expand(batch, -1, -1)
        global_tokens = torch.cat([compressed, memory], dim=1)
        global_tokens = global_tokens.reshape(batch, -1, self.num_heads, self.head_dim).transpose(1, 2)
        x_mh = x.reshape(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        attn = torch.matmul(x_mh, global_tokens.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        context = torch.matmul(attn, global_tokens)
        return context.transpose(1, 2).reshape(batch, seq, dim)

print('✅ CWAB class loaded')


In [ ]:
class StandardAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(hidden_size, hidden_size * 3)
        self.proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(x)


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_heads, num_layers, attention_class, **attn_kwargs):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm1': nn.LayerNorm(hidden_size),
                'attn': attention_class(hidden_size, num_heads, **attn_kwargs),
                'norm2': nn.LayerNorm(hidden_size),
                'ffn': nn.Sequential(
                    nn.Linear(hidden_size, hidden_size * 4),
                    nn.GELU(),
                    nn.Linear(hidden_size * 4, hidden_size)
                )
            }) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            residual = x
            x = layer['norm1'](x)
            x = layer['attn'](x)
            x = residual + x
            residual = x
            x = layer['norm2'](x)
            x = layer['ffn'](x)
            x = residual + x
        x = self.norm(x)
        return self.lm_head(x)

print('✅ Models defined')


In [ ]:
def benchmark_speed_memory(model, seq_len, device='cuda', num_iters=15):
    model = model.to(device)
    model.train()
    x = torch.randint(0, 1000, (1, seq_len)).to(device)
    for _ in range(3):
        out = model(x)
        loss = out.mean()
        loss.backward()
        model.zero_grad()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    times = []
    for _ in range(num_iters):
        start = time.time()
        out = model(x)
        loss = out.mean()
        loss.backward()
        model.zero_grad()
        if device == 'cuda':
            torch.cuda.synchronize()
        times.append(time.time() - start)
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else None
    return {'mean_time_ms': np.mean(times) * 1000, 'std_time_ms': np.std(times) * 1000, 'peak_memory_mb': peak_mem}

seq_lengths = [128, 256, 512, 1024, 2048, 4096]
configs = {'hidden_size': 256, 'num_heads': 8, 'num_layers': 4, 'vocab_size': 10000}
results = {'standard': [], 'cwab': []}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

for seq_len in tqdm(seq_lengths, desc='Benchmarking'):
    model_std = TinyTransformer(**configs, attention_class=StandardAttention, dropout=0.1)
    std_res = benchmark_speed_memory(model_std, seq_len, device)
    model_cwab = TinyTransformer(**configs, attention_class=CWAB, window_size=512, num_global_tokens=64)
    cwab_res = benchmark_speed_memory(model_cwab, seq_len, device)
    results['standard'].append(std_res)
    results['cwab'].append(cwab_res)
    del model_std, model_cwab
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

print('\n📊 Raw Benchmark Data:')
print('='*70)
print(f"{'Seq Len':>8} | {'Standard (ms)':>12} | {'CWAB (ms)':>10} | {'Speedup':>7} | {'Std Mem (MB)':>11} | {'CWAB Mem (MB)':>11}")
print('-'*70)
for i, seq in enumerate(seq_lengths):
    std_mem = f"{results['standard'][i]['peak_memory_mb']:.1f}" if results['standard'][i]['peak_memory_mb'] else 'N/A'
    cwab_mem = f"{results['cwab'][i]['peak_memory_mb']:.1f}" if results['cwab'][i]['peak_memory_mb'] else 'N/A'
    speedup = results['standard'][i]['mean_time_ms'] / results['cwab'][i]['mean_time_ms']
    print(f"{seq:8d} | {results['standard'][i]['mean_time_ms']:10.2f} ±{results['standard'][i]['std_time_ms']:.1f} | {results['cwab'][i]['mean_time_ms']:8.2f} ±{results['cwab'][i]['std_time_ms']:.1f} | {speedup:6.1f}x | {std_mem:>11} | {cwab_mem:>11}")

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.errorbar(seq_lengths, [r['mean_time_ms'] for r in results['standard']], yerr=[r['std_time_ms'] for r in results['standard']], label='Standard', capsize=3)
plt.errorbar(seq_lengths, [r['mean_time_ms'] for r in results['cwab']], yerr=[r['std_time_ms'] for r in results['cwab']], label='CWAB', capsize=3)
plt.xlabel('Sequence Length'); plt.ylabel('Time (ms)')
plt.title('Forward+Backward Time'); plt.legend(); plt.grid(True)
if device == 'cuda':
    plt.subplot(1,2,2)
    plt.plot(seq_lengths, [r['peak_memory_mb'] for r in results['standard']], 'o-', label='Standard')
    plt.plot(seq_lengths, [r['peak_memory_mb'] for r in results['cwab']], 's-', label='CWAB')
    plt.xlabel('Sequence Length'); plt.ylabel('Memory (MB)')
    plt.title('Peak Memory Usage'); plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig('honest_benchmark.png', dpi=150)
plt.show()
